# 第 05 章：Context Engineering——Runtime Context、Graph State 与 Store（离线工程实验）

**目标**：执行本章稳定公共契约并观察失败护栏。  
**环境与预计用时**：Python 3.12、offline profile，约 15–25 分钟。  
本 Notebook 由同名 Markdown 中带 `sync` 标识的实验代码生成；可复用业务逻辑始终从 `mini_deerflow` package 导入。

## 1. offline profile 初始化

显式选择离线模型档位，基础实验不得读取供应商 Key。

In [1]:
from mini_deerflow.config import ModelProfile, ModelSettings

lesson_settings = ModelSettings(profile=ModelProfile.OFFLINE)
assert lesson_settings.profile is ModelProfile.OFFLINE


## 2. 前置能力探针

验证当前 kernel 使用课程锁定的主版本，并能导入 Mini DeerFlow。

In [2]:
from importlib.metadata import version
import mini_deerflow

assert version('langchain').startswith('1.3.')
assert version('langgraph').startswith('1.2.')
assert mini_deerflow.__file__


## 3. 最小成功实验

以下单元来自 Markdown 的稳定 sync marker。

### 实验 `ch05-safe-context`

In [3]:
from mini_deerflow.context import RuntimeContext, safe_context_view

runtime_context = RuntimeContext(
    user_id="learner-1",
    workspace_root="/tmp/mini-deerflow",
    request_id="req-05-001",
    permissions=frozenset({"knowledge:read", "workspace:read"}),
    locale="zh-CN",
    auth_token="never-copy-me",
)
safe_context = safe_context_view(runtime_context)

assert safe_context["user_id"] == "learner-1"
assert safe_context["permissions"] == ["knowledge:read", "workspace:read"]
assert "auth_token" not in safe_context
assert "never-copy-me" not in repr(runtime_context)


### 实验 `ch05-checkpoint-safe-state`

In [4]:
from mini_deerflow.schemas import ArtifactRef
from mini_deerflow.state import MiddlewareTraceEvent, assert_checkpoint_safe

thread_state = {
    "messages": [],
    "artifacts": [
        ArtifactRef(path="reports/context-boundary.md", media_type="text/markdown")
    ],
    "middleware_trace": [
        MiddlewareTraceEvent(middleware="lead", hook="before_model")
    ],
}
assert_checkpoint_safe(thread_state)


### 实验 `ch05-store-cross-thread`

In [5]:
from langgraph.store.memory import InMemoryStore
from mini_deerflow.store import UserPreferenceRepository, preference_namespace

memory_store = InMemoryStore()
preferences = UserPreferenceRepository(memory_store)
preferences.save(
    "learner-1",
    {"language": "zh-CN", "citation_style": "source-first"},
)

thread_a_preferences = preferences.load("learner-1")
thread_b_preferences = preferences.load("learner-1")

assert thread_a_preferences == thread_b_preferences
assert preference_namespace("learner-1") == ("users", "learner-1")


### 实验 `ch05-store-user-isolation`

In [6]:
preferences.save("learner-2", {"language": "en-US"})

assert preferences.load("learner-1")["language"] == "zh-CN"
assert preferences.load("learner-2") == {"language": "en-US"}
assert preferences.load("unknown-user") == {}


### 实验 `ch05-thread-state-isolation`

In [7]:
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from mini_deerflow.agents import create_lead_agent
from mini_deerflow.models import create_offline_model

thread_agent = create_lead_agent(
    model=create_offline_model([
        AIMessage(content="thread-a-answer"),
        AIMessage(content="thread-b-answer"),
    ]),
    tools=[],
    checkpointer=InMemorySaver(),
)
thread_a_config = {"configurable": {"thread_id": "chapter05-thread-a"}}
thread_b_config = {"configurable": {"thread_id": "chapter05-thread-b"}}

thread_agent.invoke({"messages": [("user", "question-a")]}, config=thread_a_config)
thread_agent.invoke({"messages": [("user", "question-b")]}, config=thread_b_config)

thread_a_messages = thread_agent.get_state(thread_a_config).values["messages"]
thread_b_messages = thread_agent.get_state(thread_b_config).values["messages"]
assert "question-b" not in [message.content for message in thread_a_messages]
assert "question-a" not in [message.content for message in thread_b_messages]


### 实验 `ch05-three-boundaries`

In [8]:
from langgraph.runtime import Runtime

runtime = Runtime(context=runtime_context, store=memory_store)
current_state = {
    "messages": [],
    "middleware_trace": [
        MiddlewareTraceEvent(middleware="lead", hook="before_model")
    ],
}

context_fact = runtime.context.user_id
state_fact = current_state["middleware_trace"][0].as_text()
store_fact = UserPreferenceRepository(runtime.store).load(context_fact)["language"]

assert (context_fact, state_fact, store_fact) == (
    "learner-1",
    "lead:before_model",
    "zh-CN",
)


## 4. 状态/事件观察

观察消息、结构化对象、检索命中或 v2 event；不要只看最终自然语言。

## 5. 失败实验

失败必须被捕获并断言，证明护栏真的阻止了错误路径。

### 实验 `ch05-secret-state-failure`

In [9]:
from mini_deerflow.state import UnsafeStateError, assert_checkpoint_safe

unsafe_state = {
    "messages": [],
    "runtime": {"auth_token": "secret-value"},
}
try:
    assert_checkpoint_safe(unsafe_state)
except UnsafeStateError as error:
    secret_boundary_error = error
else:
    raise AssertionError("Secret 字段必须被 checkpoint safety guard 拒绝")

assert "auth_token" in str(secret_boundary_error)


## 6. Mini DeerFlow 工程调用

以上实验只从 `mini_deerflow` 导入公共接口；Notebook 不复制 Agent、Tool 或 Schema 实现。

## 7. 分层练习

完成同名 Markdown 的练习 A（单点修改）、B（边界判断）、C（项目扩展）和延迟回忆题。先自行作答，再运行对应 pytest 获取即时反馈。

## 8. 自动验收摘要

在项目根目录运行 `make test`。本 Notebook 的所有代码单元必须有执行计数、不得保存 error output，教程验证结果不得出现本章 drift。

## 9. 清理临时资源

当前实验使用内存对象与 `TemporaryDirectory`，退出上下文后自动清理；不要把 API Key、向量库或临时产物写回仓库。